In [1]:
#import libraries
import torch
from torch import nn


In [2]:
# setting up the structure of the NN
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2,50), nn.Tanh(),
            nn.Linear(50,50),nn.Tanh(),
            nn.Linear(50,50),nn.Tanh(),
            nn.Linear(50,50),nn.Tanh(),
            nn.Linear(50,50),nn.Tanh(),
            nn.Linear(50,1), 
        )
    def forward(self,x):
        u = self.net(x)
        return u
model = NeuralNetwork()

In [12]:
#setting up the loss
N_ic, N_b, N_f = 100, 100, 5000
convec_coeff = 30
# initial condition: t=0, x spread across [0,2pi]
x_ic = 2 * torch.pi * torch.rand(N_ic, 1)
t_ic = torch.zeros(N_ic, 1)
u_true_ic = torch.sin(x_ic)

# boundary: u(x=0)=u(x=2pi), so this goes directly in the loss section
t_bc = torch.rand(N_b, 1)


# collocation points
x_f = (2 * torch.pi * torch.rand(N_f, 1)).requires_grad_(True)
t_f = torch.rand(N_f, 1, requires_grad=True)


In [ ]:
#optimizer =  torch.optim.Adam(model.parameters(),lr=1e-3)
# loss_fn = nn.MSELoss()

# for epoch in range(10000):
#     optimizer.zero_grad()
#     u_pred_ic = model(torch.cat([x_ic,t_ic], dim=1))
#     u_pred_bc_1 = model(torch.cat([torch.zeros(N_b,1), t_bc], dim=1))
#     u_pred_bc_2 = model(torch.cat([2*torch.pi*torch.ones(N_b,1), t_bc], dim=1))
    
#     u = model(torch.cat([x_f,t_f], dim=1))
#     u_t  = torch.autograd.grad(u, t_f, torch.ones_like(u), create_graph=True)[0]
#     u_x  = torch.autograd.grad(u, x_f, torch.ones_like(u), create_graph=True)[0]
#     residual = u_t + convec_coeff * u_x

#     loss_ic = loss_fn(u_pred_ic,u_true_ic)
#     loss_bc  = loss_fn(u_pred_bc_1, u_pred_bc_2)
#     loss_pde = loss_fn(residual, torch.zeros_like(residual))
#     loss = loss_ic + loss_bc + loss_pde
    
#     loss.backward()
#     optimizer.step()
    
#     if epoch % 500 == 0:
#         print(epoch, loss.item())

0 416.20538330078125
500 0.4982692301273346
1000 0.48869431018829346
1500 0.4831734597682953
2000 0.4769163727760315
2500 0.4623716175556183
3000 0.39265450835227966
3500 0.23784352838993073
4000 0.1443498581647873
4500 0.12094836682081223
5000 0.09479046612977982
5500 0.07824012637138367
6000 0.07068786025047302
6500 0.06748510152101517
7000 0.06956902891397476
7500 0.06275881081819534
8000 0.06020369380712509
8500 0.055856965482234955
9000 0.045685917139053345
9500 0.03758695721626282


In [ ]:
# x = torch.linspace(0, 2*torch.pi, 100)
# t = torch.linspace(0, 1, 100)
# X, T = torch.meshgrid(x, t, indexing='ij')
# x_eval, t_eval = X.reshape(-1,1), T.reshape(-1,1)

# with torch.no_grad():
#     u_pred  = model(torch.cat([x_eval, t_eval], dim=1))
#     u_exact = torch.sin(x_eval - convec_coeff * t_eval)
#     error   = torch.norm(u_pred - u_exact) / torch.norm(u_exact)
# print(error.item())

0.8667882680892944


In [18]:
#Curriculum PINN Regularization
optimizer =  torch.optim.Adam(model.parameters(),lr=1e-3)
loss_fn = nn.MSELoss()
def train_model(model, optimizer, convec_coeff, epochs=1000):
    for epoch in range(epochs):
        optimizer.zero_grad()
        u_pred_ic = model(torch.cat([x_ic,t_ic], dim=1))
        u_pred_bc_1 = model(torch.cat([torch.zeros(N_b,1), t_bc], dim=1))
        u_pred_bc_2 = model(torch.cat([2*torch.pi*torch.ones(N_b,1), t_bc], dim=1))
        
        u = model(torch.cat([x_f,t_f], dim=1))
        u_t  = torch.autograd.grad(u, t_f, torch.ones_like(u), create_graph=True)[0]
        u_x  = torch.autograd.grad(u, x_f, torch.ones_like(u), create_graph=True)[0]
        residual = u_t + convec_coeff * u_x

        loss_ic = loss_fn(u_pred_ic,u_true_ic)
        loss_bc  = loss_fn(u_pred_bc_1, u_pred_bc_2)
        loss_pde = loss_fn(residual, torch.zeros_like(residual))
        loss = loss_ic + loss_bc + loss_pde
        
        loss.backward()
        optimizer.step()
    return loss.item()


def eval_error(model, convec_coeff):
    x = torch.linspace(0, 2*torch.pi, 100)
    t = torch.linspace(0, 1, 100)
    X, T = torch.meshgrid(x, t, indexing='ij')
    x_eval, t_eval = X.reshape(-1,1), T.reshape(-1,1)
    with torch.no_grad():
        u_pred  = model(torch.cat([x_eval, t_eval], dim=1))
        u_exact = torch.sin(x_eval - convec_coeff * t_eval)
        return (torch.norm(u_pred - u_exact) / torch.norm(u_exact)).item()

for beta in [1, 5, 10, 15, 20, 25, 30]:
    loss = train_model(model, optimizer, beta, epochs=10000)
    print(f"beta={beta:2d}  loss={loss:.2e}  L2={eval_error(model, beta):.2e}")

beta= 1  loss=1.17e-04  L2=2.35e-02
beta= 5  loss=2.89e-05  L2=5.66e-03
beta=10  loss=1.13e-04  L2=1.25e-02
beta=15  loss=5.48e-05  L2=1.55e-02
beta=20  loss=1.32e-03  L2=4.57e-02
beta=25  loss=1.44e-04  L2=3.32e-02
beta=30  loss=1.20e-03  L2=3.16e-02


In [20]:
import torch
from torch import nn

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2,50), nn.Tanh(),
            nn.Linear(50,50), nn.Tanh(),
            nn.Linear(50,50), nn.Tanh(),
            nn.Linear(50,50), nn.Tanh(),
            nn.Linear(50,50), nn.Tanh(),
            nn.Linear(50,1),
        )
    def forward(self, x):
        return self.net(x)


N_ic, N_b, N_f = 100, 100, 5000

x_ic = 2 * torch.pi * torch.rand(N_ic, 1)
t_ic = torch.zeros(N_ic, 1)
u_true_ic = torch.sin(x_ic)

t_bc = torch.rand(N_b, 1)

x_f = (2 * torch.pi * torch.rand(N_f, 1)).requires_grad_(True)
t_f = torch.rand(N_f, 1, requires_grad=True)

loss_fn = nn.MSELoss()


def train_model(model, convec_coeff, adam_epochs=5000, lbfgs_iters=2000):

    def compute_loss():
        u_pred_ic = model(torch.cat([x_ic, t_ic], dim=1))
        u_pred_bc_1 = model(torch.cat([torch.zeros(N_b,1), t_bc], dim=1))
        u_pred_bc_2 = model(torch.cat([2*torch.pi*torch.ones(N_b,1), t_bc], dim=1))

        u = model(torch.cat([x_f, t_f], dim=1))
        u_t = torch.autograd.grad(u, t_f, torch.ones_like(u), create_graph=True)[0]
        u_x = torch.autograd.grad(u, x_f, torch.ones_like(u), create_graph=True)[0]
        residual = u_t + convec_coeff * u_x

        loss_ic  = loss_fn(u_pred_ic, u_true_ic)
        loss_bc  = loss_fn(u_pred_bc_1, u_pred_bc_2)
        loss_pde = loss_fn(residual, torch.zeros_like(residual))
        return loss_ic + loss_bc + loss_pde

    adam = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(adam_epochs):
        adam.zero_grad()
        loss = compute_loss()
        loss.backward()
        adam.step()
    adam_loss = compute_loss().item()

    lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0,
                              max_iter=lbfgs_iters,
                              line_search_fn="strong_wolfe")

    def closure():
        lbfgs.zero_grad()
        loss = compute_loss()
        loss.backward()
        return loss

    lbfgs.step(closure)
    return adam_loss, compute_loss().item()


def eval_error(model, convec_coeff):
    x = torch.linspace(0, 2*torch.pi, 100)
    t = torch.linspace(0, 1, 100)
    X, T = torch.meshgrid(x, t, indexing='ij')
    x_eval, t_eval = X.reshape(-1,1), T.reshape(-1,1)
    with torch.no_grad():
        u_pred  = model(torch.cat([x_eval, t_eval], dim=1))
        u_exact = torch.sin(x_eval - convec_coeff * t_eval)
        return (torch.norm(u_pred - u_exact) / torch.norm(u_exact)).item()


model = NeuralNetwork()

for beta in [1, 5, 10, 15, 20, 25, 30]:
    adam_loss, final_loss = train_model(model, beta)
    print(f"beta={beta:2d}  adam={adam_loss:.2e}  lbfgs={final_loss:.2e}  "
          f"L2={eval_error(model, beta):.2e}")

beta= 1  adam=2.69e-04  lbfgs=1.94e-05  L2=5.07e-03
beta= 5  adam=6.55e-04  lbfgs=2.08e-06  L2=8.77e-04
beta=10  adam=9.55e-05  lbfgs=7.26e-06  L2=2.06e-03
beta=15  adam=2.67e-03  lbfgs=1.88e-05  L2=8.03e-03
beta=20  adam=4.45e-02  lbfgs=1.20e-05  L2=5.23e-03
beta=25  adam=1.86e-02  lbfgs=1.85e-05  L2=1.04e-02
beta=30  adam=1.47e-03  lbfgs=4.71e-05  L2=1.68e-02
